[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Classification &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, roc_auc_score

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' labels and models, rebuilt here from the same made-up stations so that each answer
runs on its own.

In [2]:
frost_train = (train["tmin"] < 0).astype(int)
frost_valid = (valid["tmin"] < 0).astype(int)


def bands(rows):
    return pd.cut(rows["tmin"], [-np.inf, -10, 0, np.inf], right=False, labels=False)


lightgbm_frost = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_frost.fit(train[FEATURES], frost_train)
lightgbm_bands = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_bands = lightgbm_bands.fit(train[FEATURES], bands(train))

**1.** Turn LightGBM's probabilities of frost into labels at thresholds of 0.3 and 0.7 as well as 0.5, and print the accuracy on the validation stations at each.

In [3]:
probability = lightgbm_frost.predict_proba(valid[FEATURES])[:, 1]
for threshold in (0.3, 0.5, 0.7):
    print(threshold, round(accuracy_score(frost_valid, (probability > threshold).astype(int)), 3))

0.3 0.948
0.5 0.945
0.7 0.942


Accuracy is 0.948 at 0.3, 0.945 at 0.5 and 0.942 at 0.7. Most probabilities are close to 0 or 1, so few days change label between these thresholds.

**2.** Train an `LGBMClassifier` for frost on `cloud_cover`, `wind_ms` and `day_of_year` alone, under a new name, and print its log loss and AUC on the validation stations beside the eight-feature model's.

In [4]:
three = ["cloud_cover", "wind_ms", "day_of_year"]
small = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=0, verbose=-1)
small.fit(train[three], frost_train)
for name, model, columns in (("three features", small, three), ("eight features", lightgbm_frost, FEATURES)):
    probability = model.predict_proba(valid[columns])[:, 1]
    print(f"{name}: log loss {log_loss(frost_valid, probability):.3f} | AUC {roc_auc_score(frost_valid, probability):.3f}")

three features: log loss 0.214 | AUC 0.972
eight features: log loss 0.146 | AUC 0.989


Three features give a log loss of 0.214 and an AUC of 0.972, against 0.146 and 0.989 with all eight. The cloud, the wind and the season already order the days well, and the other five features lower the log loss further.

**3.** Add a band of cold below −20 to the bands, in a function of your own, leaving `bands` and `bands_train` as they are, train an `XGBClassifier` on them, and print the number of days in each band, the number of trees and the accuracy on the validation stations.

In [5]:
def four_bands(rows):
    return pd.cut(rows["tmin"], [-np.inf, -20, -10, 0, np.inf], right=False, labels=False)

four = xgb.XGBClassifier(n_estimators=300, learning_rate=0.1, random_state=0)
four.fit(train[FEATURES], four_bands(train))
print("days in each band:", np.bincount(four_bands(train)))
print("trees:", len(four.get_booster().get_dump()))
print("accuracy:", round(accuracy_score(four_bands(valid), four.predict(valid[FEATURES])), 3))

days in each band: [  28 1573 6619 6780]
trees: 1200
accuracy: 0.902


The new band below −20 has only 28 training days. Four bands and 300 rounds make 1,200 trees, a tree for each band every round, and the accuracy on the validation stations is 0.902.

**4.** For the first three validation days, print the raw scores `lightgbm_bands` gives each band with `raw_score=True`, and check that the softmax of the raw scores, e^score divided by the sum of e^score over the bands, gives `predict_proba`.

In [6]:
raw_scores = lightgbm_bands.predict(valid[FEATURES].iloc[:3], raw_score=True)
softmax = np.exp(raw_scores) / np.exp(raw_scores).sum(axis=1, keepdims=True)
print(raw_scores.round(3))
print(np.allclose(softmax, lightgbm_bands.predict_proba(valid[FEATURES].iloc[:3])))

[[-11.824   0.258   0.194]
 [ -7.773   1.483  -6.613]
 [-12.547  -0.117   1.426]]
True


The raw scores can be any number, such as −11.824 for the coldest band on the first day, and the softmax of the raw scores matches `predict_proba`, which `np.allclose` confirms by printing `True`.

---

&#8592; **Back to:** [Classification](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/10-classification.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
